# Mémoire conversationnelle LangChain
Dernière mise à jour : 28 juillet 2025

Défi quotidien : Démarrer avec la mémoire conversationnelle LangChain


Objectif
Dans cet exercice, vous construisez l'élément fondamental d'un chatbot contextuel : une mémoire tampon qui enregistre et rappelle les échanges passés afin que le bot puisse se souvenir de ce qui a été dit précédemment. Plutôt que de traiter chaque message utilisateur isolément, vous connecterez la mémoire tampon de conversation de LangChain à :

Importez et initialisez un objet mémoire depuis LangChain.
Capturez chaque tour du dialogue (à la fois les entrées utilisateur et les sorties du bot) en appelant memory.save_context(…) deux fois (une fois pour l'échange de salutations, une fois pour la demande de blague).
Récupérez l'historique complet avec memory.load_memory_variables({}), qui renvoie un journal structuré de tous les messages enregistrés.
À la fin, vous aurez démontré comment préserver deux tours de conversation dans un script Python simple. Ce même modèle (importation d'une classe mémoire, enregistrement de chaque interaction et chargement du contexte accumulé) peut ensuite être intégré à n'importe quel appel LLM en aval afin que votre chatbot puisse délivrer des conversations multitours véritablement cohérentes.



👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Comment préserver le contexte dans les conversations IA à l'aide de LangChain
Comment utiliser ConversationBufferMemorypour stocker et récupérer l'historique des dialogues
Comment implémenter la mémoire dans une interaction simple avec un chatbot


🛠️ Ce que vous allez créer
Une mémoire tampon conversationnelle qui stocke deux échanges utilisateur-bot et récupère l'historique complet des conversations

# Analyse approfondie — LangChain Conversational Memory

## Résumé exécutif

L’exercice vise à montrer comment **stocker et relire un historique de dialogue** avec `ConversationBufferMemory` : deux tours (salutation + demande de blague), puis restitution du contexte complet. Techniquement, on utilise `save_context()` pour journaliser les entrées/sorties et `load_memory_variables({})` pour récupérer l’historique. Point important en 2025 : `ConversationBufferMemory` est **déprécié depuis LangChain v0.3.1** au profit des primitives « message history » via **LangGraph** ou `RunnableWithMessageHistory`. Le code legacy fonctionne encore mais n’est plus recommandé pour de nouveaux projets. ([LangChain][1])

---

## Ce que l’exercice démontre (et ce que ça implique)

1. **Initialiser une mémoire** capable d’agréger des tours de conversation.
   → `ConversationBufferMemory()` en « legacy » (toujours utilisable). ([LangChain][1])
2. **Journaliser chaque tour** (user + bot) avec `save_context(inputs, outputs)`.
   → Les clés par défaut attendues sont `input` et `output`, sauf si vous les changez. ([LangChain][1])
3. **Relire l’historique** via `load_memory_variables({})` qui renvoie un dict avec la clé d’historique (par défaut `history` dans les intégrations de chaînes). **Si `return_messages=False`** on obtient une **chaîne**, **si `True`** une **liste de `BaseMessage`**. ([LangChain][1])
4. **Brancher cette mémoire** plus tard sur un LLM ou une chaîne pour obtenir des réponses multi-tours cohérentes. Dans v0.3+, LangChain recommande **LangGraph** (état = liste de messages, checkpointer, threads). ([LangChain][2])

---

## API utile (legacy) et comportements par défaut

* **Méthodes clés** :
  `save_context(inputs: dict, outputs: dict)` • `load_memory_variables(inputs: dict) -> dict` • `clear()`. ([LangChain][1])
* **Options fréquentes** :
  `return_messages: bool` (False = historique concaténé en texte, True = liste de messages), `input_key`, `output_key`, `ai_prefix`, `human_prefix`. ([LangChain][1])
* **Remarque v0.3+** : la doc de migration montre l’usage avec un `memory_key="chat_history"` lorsque mémoire et prompt sont reliés via `LLMChain`/`ConversationChain`. ([LangChain][3])

---

## Pièges fréquents (et corrections)

* **Clés d’E/S incohérentes** : si vous logguez `{"question": ...}` mais l’output s’appelle `{"output": ...}`, configurez `input_key`/`output_key` ou normalisez vos dicts. ([LangChain][1])
* **Historique trop long** : `ConversationBufferMemory` grossit sans limite → coûts/contexte. Utiliser **`ConversationTokenBufferMemory(max_token_limit=...)`** ou **`ConversationSummaryBufferMemory`** (résume + fenêtre récente). Tous deux sont aussi dépréciés en 0.3.1 mais illustrent les stratégies; préférez LangGraph pour les nouveaux projets. ([api.python.langchain.com][4], [LangChain][5])
* **Threads/sessions** : un seul objet mémoire ≠ plusieurs conversations. Pour gérer plusieurs fils, créez **une instance par thread** ou migrez vers **LangGraph + `MemorySaver`** qui isole par `thread_id`. ([LangChain][3])
* **Persistance inter-sessions** : `ConversationBufferMemory` est volatile. Pour persister, utilisez **LangGraph checkpointers** (mémoire, SQLite, Postgres, Redis). ([LangChain][2])



# **Notebook Jupyter** fonctionnel :

* fonctionne en local avec Ollama (exemple avec `llama3`),
* explique chaque étape,
* affiche clairement l’historique de conversation,
* intègre les bonnes pratiques.

---

```python
# Cellule 1 : Installation des dépendances nécessaires (si pas déjà installées)

# À exécuter UNE FOIS. Tu peux ignorer si c'est déjà installé dans ton environnement.
!pip install -U langchain_community langgraph
```

---

```python
# Cellule 2 : Import des modules

from langchain_community.chat_models import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_core.messages import HumanMessage
```

---

```python
# Cellule 3 : Initialisation du modèle Ollama

# Assure-toi que le serveur Ollama tourne (commande : ollama serve)
# Le modèle "llama3" doit être téléchargé au préalable : ollama pull llama3

llm = ChatOllama(model="llama3")  # Change si besoin (phi3, mistral...)
```

---

```python
# Cellule 4 : Construction du workflow LangGraph (mémoire + graphe)

workflow = StateGraph(state_schema=MessagesState)

def call_model(state: MessagesState):
    # Appelle le modèle sur l'historique des messages
    resp = llm.invoke(state["messages"])
    return {"messages": resp}

workflow.add_node("model", call_model)
workflow.add_edge(START, "model")
app = workflow.compile(checkpointer=MemorySaver())

cfg = {"configurable": {"thread_id": "conv-ollama"}}
```

---

```python
# Cellule 5 : Simuler deux tours de conversation

# Premier échange
app.invoke({"messages": [HumanMessage("Hello, how are you?")]}, cfg)

# Second échange
app.invoke({"messages": [HumanMessage("Tell me a joke.")]}, cfg)
```

---

```python
# Cellule 6 : Récupérer et afficher l’historique complet

state = app.get_state(cfg).values

print("=== Historique de conversation ===")
for i, m in enumerate(state["messages"], 1):
    print(f"{i:02d} | {m.type.upper():<6} : {m.content}")
```

**Exemple d’affichage attendu :**

```
=== Historique de conversation ===
01 | HUMAN  : Hello, how are you?
02 | AI     : I'm just a language model, I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have! How can I help you today?
03 | HUMAN  : Tell me a joke.
04 | AI     : Here's one:
              Why couldn't the bicycle stand up by itself?
              (Wait for it...)
              Because it was two-tired!
              Hope that made you smile! Do you want to hear another one?
```

---

```python
# Cellule 7 : (Optionnel) - Ajouter d'autres échanges et réafficher l'historique

app.invoke({"messages": [HumanMessage("Encore une blague, mais en français !")]}, cfg)

state = app.get_state(cfg).values

print("=== Historique mis à jour ===")
for i, m in enumerate(state["messages"], 1):
    print(f"{i:02d} | {m.type.upper():<6} : {m.content}")
```


# Copie output Terminal

Mes cours
Calendrier
Classement
Offres d'emploi
À propos de DI
Conditions générales
Confidentialité
Menu
Mes cours
Mes réalisations
Mon trophée
Mon diplôme
 Parrainer un ami
Mon paiement
Comparateur de CV
Salles P2P
Mon OctoHelp
✨ Assistant IA
 Bootcamp GenAI et Machine Learning 2025 - Temps plein 2025 - PSTB Ingénierie rapide Mini projet : Intégrer la mémoire et l'intelligence aux chatbots IA Mémoire conversationnelle LangChain
Mémoire conversationnelle LangChain
Dernière mise à jour : 28 juillet 2025

Défi quotidien : Démarrer avec la mémoire conversationnelle LangChain


Objectif
Dans cet exercice, vous construisez l'élément fondamental d'un chatbot contextuel : une mémoire tampon qui enregistre et rappelle les échanges passés afin que le bot puisse se souvenir de ce qui a été dit précédemment. Plutôt que de traiter chaque message utilisateur isolément, vous connecterez la mémoire tampon de conversation de LangChain à :

Importez et initialisez un objet mémoire depuis LangChain.
Capturez chaque tour du dialogue (à la fois les entrées utilisateur et les sorties du bot) en appelant memory.save_context(…) deux fois (une fois pour l'échange de salutations, une fois pour la demande de blague).
Récupérez l'historique complet avec memory.load_memory_variables({}), qui renvoie un journal structuré de tous les messages enregistrés.
À la fin, vous aurez démontré comment préserver deux tours de conversation dans un script Python simple. Ce même modèle (importation d'une classe mémoire, enregistrement de chaque interaction et chargement du contexte accumulé) peut ensuite être intégré à n'importe quel appel LLM en aval afin que votre chatbot puisse délivrer des conversations multitours véritablement cohérentes.



👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Comment préserver le contexte dans les conversations IA à l'aide de LangChain
Comment utiliser ConversationBufferMemorypour stocker et récupérer l'historique des dialogues
Comment implémenter la mémoire dans une interaction simple avec un chatbot


🛠️ Ce que vous allez créer
Une mémoire tampon conversationnelle qui stocke deux échanges utilisateur-bot et récupère l'historique complet des conversations
Votre tâche
1. Importez le module de mémoire

Astuce : vous recherchez une classe appelée ConversationBufferMemorydans le module de mémoire LangChain.
2. Initialiser la mémoire

Astuce : créez une instance de la classe mémoire pour commencer à stocker le contexte.
3. Simulez une première interaction

Enregistrez l'échange suivant :
Utilisateur:"Hello, how are you?"
Bot:"I'm fine, thank you. How can I assist you today?"
Astuce : utilisez une méthode appelée save_contextavec des dictionnaires pour inputet output.
4. Simulez un message de suivi

Enregistrer un autre échange :
Utilisateur:"Tell me a joke."
Bot:"Why did the chicken cross the road? To get to the other side."
Astuce : utilisez à nouveau la même méthode pour ajouter ce contexte.
5. Récupérer l'historique des conversations

Chargez tout le contexte stocké dans une variable appelée conversation_history.
Astuce : il existe une méthode appelée load_memory_variablesqui prend un dictionnaire vide comme paramètre.
6. Imprimer la mémoire (facultatif)

Imprimez conversation_historypour voir comment LangChain stocke les messages précédents.


Durée et difficulté
Durée (environ)	Difficulté
2 heures	⭐⭐
Soumettez votre défi quotidien
Une fois terminé, téléchargez votre Jupyter Notebook contenant votre code, vos visualisations et vos informations sur GitHub.


© 2025 Developers Institute . Tous droits réservés.

 
 
 
Mini projet : Intégrer la mémoire et l'intelligence aux chatbots IA
0% terminé
Notes de cours
Exercices
Défi quotidien
Mémoire conversationnelle LangChain
15 XP  
